# 13 — Evaluate BCN+MNH Cancer-Risk CNN on Frozen BCN20000 Test Set

**Issue:** D4.6 (#142)  
**Depends on:** D4.5 (#141)  
**Purpose:** Evaluate the BCN+MNH retrained CNN on the **identical** frozen BCN20000 test set used in #120, producing a direct side-by-side comparison with the BCN-only baseline. This is the only valid measurement of whether the MNH augmentation actually improves melanoma recall.

**Framework / approach (matches D4.5 decision):** standalone PyTorch notebook. Loads `models/bcn_mnh_cancer_risk_effnet_b0/best_model.pth` directly, runs inference with `get_eval_transforms(224)` from `src/data/transforms.py` (the same module #120 used), computes metrics from scratch, and writes to D4.6-specific output paths so the BCN-only baseline JSON is **not** overwritten.

**Notebook numbering:** spec calls this `notebooks/10_evaluate_bcn_mnh_cancer_risk_cnn.ipynb`, but `10` was taken by D4.3 (mapping). Continuing the sequence as `13_…`.

**Communication rules (DEC-006, #142):** no clinical diagnosis claims; do not claim all cancers detected; framing is an educational prototype throughout.

## 1. Verify frozen test set integrity

In [1]:
import hashlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("/Users/rehmaaziz/revela")
sys.path.insert(0, str(ROOT))  # so we can `import src.*`

BCN_TEST_CSV     = ROOT / "data/processed/bcn20000_cancer_risk/test.csv"
BASELINE_JSON    = ROOT / "outputs/metrics/bcn20000_cancer_risk_test_metrics.json"
BASELINE_REPORT  = ROOT / "outputs/metrics/bcn20000_cancer_risk_classification_report.csv"
MODEL_DIR        = ROOT / "models/bcn_mnh_cancer_risk_effnet_b0"
MODEL_CKPT       = MODEL_DIR / "best_model.pth"
CLASS_IDX_PATH   = MODEL_DIR / "class_to_idx.json"

OUT_METRICS_JSON   = ROOT / "outputs/metrics/bcn_mnh_cancer_risk_test_metrics.json"
OUT_REPORT_CSV     = ROOT / "outputs/metrics/bcn_mnh_classification_report.csv"
OUT_CM_PNG         = ROOT / "outputs/plots/bcn_mnh_confusion_matrix.png"
OUT_SUMMARY_TXT    = ROOT / "outputs/metrics/bcn_mnh_evaluation_summary.txt"

EXPECTED_TEST_HASH = "a67861586e00812aadf46f2bdb4bc01b"  # logged in #140, #141

def file_hash(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()

test_hash_before = file_hash(BCN_TEST_CSV)
test_df = pd.read_csv(BCN_TEST_CSV)

print(f"Test set path:  {BCN_TEST_CSV}")
print(f"Test set rows:  {len(test_df):,}")
print(f"Test set hash:  {test_hash_before}")
print(f"Expected hash:  {EXPECTED_TEST_HASH}")
assert test_hash_before == EXPECTED_TEST_HASH, \
    f"Hash mismatch! Test set is not the one used in #120 / #141. STOP."
print("\nHash matches #120 baseline — comparison will be apples-to-apples.")

print("\nClass distribution in test set:")
for cls, n in test_df["class_label"].value_counts().items():
    print(f"  {cls:45s} {n:>5,}  ({n/len(test_df)*100:.1f}%)")

Test set path:  /Users/rehmaaziz/revela/data/processed/bcn20000_cancer_risk/test.csv
Test set rows:  2,659
Test set hash:  a67861586e00812aadf46f2bdb4bc01b
Expected hash:  a67861586e00812aadf46f2bdb4bc01b

Hash matches #120 baseline — comparison will be apples-to-apples.

Class distribution in test set:
  Benign nevus                                    935  (35.2%)
  Non-melanoma skin cancer                        656  (24.7%)
  Melanoma                                        572  (21.5%)
  Other non-cancer / indeterminate lesion         496  (18.7%)


## 2. Load BCN-only baseline metrics from #120

In [2]:
with BASELINE_JSON.open() as fh:
    baseline = json.load(fh)

print("BCN-only baseline (#120):")
for k, v in baseline.items():
    if isinstance(v, float):
        print(f"  {k:35s} {v:.4f}")
    elif isinstance(v, int):
        print(f"  {k:35s} {v}")

# Per-class baseline P/R/F1 (for class-wise delta)
baseline_report = pd.read_csv(BASELINE_REPORT)
print("\nBaseline class-wise report:")
print(baseline_report.to_string(index=False))

BCN-only baseline (#120):
  top1_accuracy                       0.6777
  macro_f1                            0.6581
  balanced_accuracy                   0.6585
  cancer_recall                       0.7191
  cancer_fnr                          0.2809
  melanoma_recall                     0.5787
  nmsc_recall                         0.7241
  num_test_examples                   2659
  checkpoint_epoch                    6
  checkpoint_val_macro_f1             0.6924

Baseline class-wise report:
                             class_name  precision   recall       f1  support
                               Melanoma   0.605119 0.578671 0.591600      572
               Non-melanoma skin cancer   0.728528 0.724085 0.726300      656
                           Benign nevus   0.767991 0.764706 0.766345      935
Other non-cancer / indeterminate lesion   0.531191 0.566532 0.548293      496


## 3. Load BCN+MNH model and run inference (PyTorch, MPS)

Uses `get_eval_transforms(224)` from `src/data/transforms.py` — same preprocessing module #120 used. Architecture rebuilt via `create_model(num_classes=4)` and weights loaded from the D4.5 checkpoint.

In [3]:
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from src.data.transforms import get_eval_transforms
from src.model.model import create_model

with CLASS_IDX_PATH.open() as fh:
    class_to_idx = {k: int(v) for k, v in json.load(fh).items()}
idx_to_class = {v: k for k, v in class_to_idx.items()}
CLASS_NAMES = [idx_to_class[i] for i in range(len(class_to_idx))]
print(f"class_to_idx: {class_to_idx}")
print(f"CLASS_NAMES (in idx order): {CLASS_NAMES}")

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Device: {device}")

model = create_model(num_classes=len(CLASS_NAMES), pretrained=False)
ckpt  = torch.load(MODEL_CKPT, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval().to(device)
print(f"Loaded checkpoint: epoch={ckpt['epoch']} val_macro_f1={ckpt['val_macro_f1']:.4f}")

class_to_idx: {'Melanoma': 0, 'Non-melanoma skin cancer': 1, 'Benign nevus': 2, 'Other non-cancer / indeterminate lesion': 3}
CLASS_NAMES (in idx order): ['Melanoma', 'Non-melanoma skin cancer', 'Benign nevus', 'Other non-cancer / indeterminate lesion']
Device: mps
Loaded checkpoint: epoch=6 val_macro_f1=0.6768


In [4]:
class BCNTestDataset(Dataset):
    """Minimal dataset for inference on BCN20000 test CSV. Returns (image_tensor, true_idx)."""
    def __init__(self, df: pd.DataFrame, class_to_idx: dict, root: Path, transform):
        self.df = df.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.root = root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root / row["image_path"]
        with Image.open(img_path) as im:
            img = im.convert("RGB")
        return self.transform(img), self.class_to_idx[row["class_label"]]

test_dataset = BCNTestDataset(test_df, class_to_idx, ROOT, get_eval_transforms(224))
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)
assert len(test_dataset) == len(test_df)

y_true_list = []
y_pred_list = []
y_prob_list = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = probs.argmax(axis=1)
        y_true_list.extend(labels.numpy().tolist())
        y_pred_list.extend(preds.tolist())
        y_prob_list.append(probs)

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)
y_prob = np.concatenate(y_prob_list, axis=0)
assert len(y_pred) == len(test_df), "Prediction count mismatch"
print(f"\nInference complete. {len(y_pred):,} predictions on {len(test_df):,} test images.")


Inference complete. 2,659 predictions on 2,659 test images.


## 4. Compute all required metrics

In [5]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

report     = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
cm_count   = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
cm_norm    = cm_count.astype(float) / cm_count.sum(axis=1, keepdims=True)

top1_acc     = accuracy_score(y_true, y_pred)
macro_f1     = f1_score(y_true, y_pred, average="macro", zero_division=0)
balanced_acc = balanced_accuracy_score(y_true, y_pred)

mel_idx  = class_to_idx["Melanoma"]
nmsc_idx = class_to_idx["Non-melanoma skin cancer"]

mel_recall    = report["Melanoma"]["recall"]
mel_precision = report["Melanoma"]["precision"]
mel_f1        = report["Melanoma"]["f1-score"]
mel_support   = int(report["Melanoma"]["support"])
mel_fnr       = 1.0 - mel_recall   # primary safety metric

nmsc_recall = report["Non-melanoma skin cancer"]["recall"]

# Combined cancer recall: true cancer (Mel or NMSC) predicted as ANY cancer class
cancer_tp    = cm_count[mel_idx, mel_idx] + cm_count[mel_idx, nmsc_idx] + cm_count[nmsc_idx, mel_idx] + cm_count[nmsc_idx, nmsc_idx]
cancer_total = cm_count[mel_idx, :].sum() + cm_count[nmsc_idx, :].sum()
cancer_recall = cancer_tp / cancer_total
cancer_fnr    = 1.0 - cancer_recall

print(f"\nBCN+MNH results on frozen test set ({len(test_df):,} images):")
print(f"  Top-1 accuracy:         {top1_acc:.4f}")
print(f"  Macro-F1:               {macro_f1:.4f}")
print(f"  Balanced accuracy:      {balanced_acc:.4f}")
print(f"  ── Melanoma (n={mel_support}) ──────────────────")
print(f"  Melanoma recall:        {mel_recall:.4f}    ← primary objective of MNH augmentation")
print(f"  Melanoma FNR:           {mel_fnr:.4f}    ← ~1 in {1/mel_fnr:.0f} melanoma cases missed (primary safety metric)")
print(f"  Melanoma precision:     {mel_precision:.4f}")
print(f"  Melanoma F1:            {mel_f1:.4f}")
print(f"  ── Other ────────────────────────────────────")
print(f"  NMSC recall:            {nmsc_recall:.4f}")
print(f"  Cancer / malignant recall (Mel+NMSC → any cancer class): {cancer_recall:.4f}")
print(f"  Cancer FNR:             {cancer_fnr:.4f}")


BCN+MNH results on frozen test set (2,659 images):
  Top-1 accuracy:         0.6762
  Macro-F1:               0.6552
  Balanced accuracy:      0.6571
  ── Melanoma (n=572) ──────────────────
  Melanoma recall:        0.6136    ← primary objective of MNH augmentation
  Melanoma FNR:           0.3864    ← ~1 in 3 melanoma cases missed (primary safety metric)
  Melanoma precision:     0.6405
  Melanoma F1:            0.6268
  ── Other ────────────────────────────────────
  NMSC recall:            0.7530
  Cancer / malignant recall (Mel+NMSC → any cancer class): 0.7370
  Cancer FNR:             0.2630


## 5. Delta table: BCN+MNH vs BCN-only (#120)

In [6]:
metrics_new = {
    "top1_accuracy":    top1_acc,
    "macro_f1":         macro_f1,
    "balanced_accuracy":balanced_acc,
    "melanoma_recall":  mel_recall,
    "melanoma_fnr":     mel_fnr,
    "nmsc_recall":      nmsc_recall,
    "cancer_recall":    cancer_recall,
    "cancer_fnr":       cancer_fnr,
}

# melanoma_fnr is derived from baseline.melanoma_recall (1 - recall)
baseline_full = dict(baseline)
baseline_full["melanoma_fnr"] = 1.0 - baseline["melanoma_recall"]

print(f"\n{'Metric':<25} {'BCN-only':>10} {'BCN+MNH':>10} {'Delta':>10} {'Status':>8}")
print("-" * 70)
for k, v_new in metrics_new.items():
    v_base = baseline_full.get(k)
    if v_base is None:
        continue
    delta = v_new - v_base
    # FNR metrics: lower is better. Recall/accuracy/F1: higher is better.
    is_fnr = k.endswith("_fnr")
    better = (delta < -0.005) if is_fnr else (delta > 0.005)
    worse  = (delta > 0.005)  if is_fnr else (delta < -0.005)
    status = "GAIN" if better else ("REGR" if worse else "flat")
    print(f"{k:<25} {v_base:>10.4f} {v_new:>10.4f} {delta:>+10.4f} {status:>8}")


Metric                      BCN-only    BCN+MNH      Delta   Status
----------------------------------------------------------------------
top1_accuracy                 0.6777     0.6762    -0.0015     flat
macro_f1                      0.6581     0.6552    -0.0030     flat
balanced_accuracy             0.6585     0.6571    -0.0014     flat
melanoma_recall               0.5787     0.6136    +0.0350     GAIN
melanoma_fnr                  0.4213     0.3864    -0.0350     GAIN
nmsc_recall                   0.7241     0.7530    +0.0290     GAIN
cancer_recall                 0.7191     0.7370    +0.0179     GAIN
cancer_fnr                    0.2809     0.2630    -0.0179     GAIN


## 6. Class-wise breakdown — BCN+MNH vs BCN-only

In [7]:
rows = []
for cls in CLASS_NAMES:
    r_new = report[cls]
    base_row = baseline_report[baseline_report["class_name"] == cls].iloc[0]
    rows.append({
        "class":             cls,
        "precision_BCN":     float(base_row["precision"]),
        "precision_BCN+MNH": r_new["precision"],
        "recall_BCN":        float(base_row["recall"]),
        "recall_BCN+MNH":    r_new["recall"],
        "f1_BCN":            float(base_row["f1"]),
        "f1_BCN+MNH":        r_new["f1-score"],
        "support":           int(r_new["support"]),
    })

cw = pd.DataFrame(rows)
cw["recall_delta"] = cw["recall_BCN+MNH"] - cw["recall_BCN"]
cw["f1_delta"]     = cw["f1_BCN+MNH"]     - cw["f1_BCN"]
cw["precision_delta"] = cw["precision_BCN+MNH"] - cw["precision_BCN"]

print("\nClass-wise comparison (BCN+MNH vs BCN-only #120):")
print(cw[["class", "support", "recall_BCN", "recall_BCN+MNH", "recall_delta",
         "precision_BCN+MNH", "f1_BCN+MNH", "f1_delta"]].to_string(index=False, float_format="{:.4f}".format))


Class-wise comparison (BCN+MNH vs BCN-only #120):
                                  class  support  recall_BCN  recall_BCN+MNH  recall_delta  precision_BCN+MNH  f1_BCN+MNH  f1_delta
                               Melanoma      572      0.5787          0.6136        0.0350             0.6405      0.6268    0.0352
               Non-melanoma skin cancer      656      0.7241          0.7530        0.0290             0.6919      0.7212   -0.0051
                           Benign nevus      935      0.7647          0.7455       -0.0193             0.8030      0.7732    0.0068
Other non-cancer / indeterminate lesion      496      0.5665          0.5161       -0.0504             0.4839      0.4995   -0.0488


## 7. Confusion matrices (count + row-normalized)

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

OUT_CM_PNG.parent.mkdir(parents=True, exist_ok=True)
labels_short = ["Melanoma", "NMSC", "Benign\nnevus", "Other"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm_count, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_short, yticklabels=labels_short, ax=axes[0])
axes[0].set_title("BCN+MNH — Confusion Matrix (Count)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("True label"); axes[0].set_xlabel("Predicted label")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=labels_short, yticklabels=labels_short, ax=axes[1])
axes[1].set_title("BCN+MNH — Confusion Matrix (Row-Normalized)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("True label"); axes[1].set_xlabel("Predicted label")

plt.suptitle(f"BCN+MNH Evaluation | Melanoma Recall: {mel_recall:.1%} | FNR: {mel_fnr:.1%} | Test n={len(test_df):,}",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(OUT_CM_PNG, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {OUT_CM_PNG}")

Saved: /Users/rehmaaziz/revela/outputs/plots/bcn_mnh_confusion_matrix.png


## 8. Misclassification analysis — where do missed melanoma go?

Note: BCN20000 frozen test set is exclusively BCN-sourced (no MNH images per D4.4 design — MNH was merged into the training pool only). A source-split analysis is not applicable here; instead we look at which classes melanoma false negatives get predicted as.

In [9]:
ann = test_df.copy().reset_index(drop=True)
ann["y_true"] = y_true
ann["y_pred"] = y_pred
ann["correct"] = y_true == y_pred

mel_rows = ann[ann["y_true"] == mel_idx]
mel_correct   = int(mel_rows["correct"].sum())
mel_incorrect = int((~mel_rows["correct"]).sum())
print(f"Melanoma test cases: {len(mel_rows):,}")
print(f"  Correctly classified: {mel_correct:,} ({mel_correct/len(mel_rows)*100:.1f}%)")
print(f"  Missed (false negative): {mel_incorrect:,} ({mel_incorrect/len(mel_rows)*100:.1f}%)\n")

print("Missed melanoma cases predicted as:")
fn_rows = mel_rows[~mel_rows["correct"]]
for pred_idx, count in fn_rows["y_pred"].value_counts().items():
    name = idx_to_class[pred_idx]
    pct  = count / len(fn_rows) * 100
    print(f"  {name:45s} {count:>4,}  ({pct:>5.1f}% of FN)")

# Compare melanoma FN counts: BCN+MNH vs BCN-only baseline
baseline_mel_fn = int(round((1 - baseline["melanoma_recall"]) * mel_support))
print(f"\nMelanoma false negatives — BCN-only baseline: {baseline_mel_fn:,} ({(1-baseline['melanoma_recall'])*100:.1f}%)")
print(f"Melanoma false negatives — BCN+MNH:           {mel_incorrect:,} ({mel_fnr*100:.1f}%)")
print(f"Change in missed melanoma:                    {mel_incorrect - baseline_mel_fn:+,} cases")

Melanoma test cases: 572
  Correctly classified: 351 (61.4%)
  Missed (false negative): 221 (38.6%)

Missed melanoma cases predicted as:
  Benign nevus                                    93  ( 42.1% of FN)
  Other non-cancer / indeterminate lesion         86  ( 38.9% of FN)
  Non-melanoma skin cancer                        42  ( 19.0% of FN)

Melanoma false negatives — BCN-only baseline: 241 (42.1%)
Melanoma false negatives — BCN+MNH:           221 (38.6%)
Change in missed melanoma:                    -20 cases


## 9. Save all outputs

In [10]:
OUT_METRICS_JSON.parent.mkdir(parents=True, exist_ok=True)

full_metrics = {
    **metrics_new,
    "melanoma_precision":    mel_precision,
    "melanoma_f1":           mel_f1,
    "melanoma_support":      mel_support,
    "num_test_examples":     int(len(test_df)),
    "test_set_hash":         test_hash_before,
    "test_csv_path":         str(BCN_TEST_CSV),
    "model_checkpoint":      str(MODEL_CKPT),
    "checkpoint_epoch":      int(ckpt["epoch"]),
    "checkpoint_val_macro_f1": float(ckpt["val_macro_f1"]),
    "baseline_issue":        "#120",
    "baseline_metrics":      baseline,
    "class_names":           CLASS_NAMES,
    "class_to_idx":          class_to_idx,
    "classification_report": report,
    "confusion_matrix":      cm_count.tolist(),
    "confusion_matrix_normalized": cm_norm.tolist(),
}
with OUT_METRICS_JSON.open("w") as fh:
    json.dump(full_metrics, fh, indent=2)
print(f"Saved: {OUT_METRICS_JSON}")

report_rows = []
for cls in CLASS_NAMES + ["macro avg", "weighted avg"]:
    r = report.get(cls, {})
    report_rows.append({
        "class_name": cls,
        "precision":  r.get("precision", ""),
        "recall":     r.get("recall", ""),
        "f1":         r.get("f1-score", ""),
        "support":    r.get("support", ""),
    })
pd.DataFrame(report_rows).to_csv(OUT_REPORT_CSV, index=False)
print(f"Saved: {OUT_REPORT_CSV}")

# Final hash check — test set must not have changed during inference
test_hash_after = file_hash(BCN_TEST_CSV)
assert test_hash_after == test_hash_before, \
    f"Test file changed during inference! before={test_hash_before} after={test_hash_after}"
print(f"\nBCN test hash unchanged: {test_hash_after}")

Saved: /Users/rehmaaziz/revela/outputs/metrics/bcn_mnh_cancer_risk_test_metrics.json
Saved: /Users/rehmaaziz/revela/outputs/metrics/bcn_mnh_classification_report.csv

BCN test hash unchanged: a67861586e00812aadf46f2bdb4bc01b


## 10. Numeric summary paragraph (no clinical claims)

In [11]:
mel_delta = mel_recall - baseline["melanoma_recall"]
fnr_delta = mel_fnr    - (1 - baseline["melanoma_recall"])
f1_delta  = macro_f1   - baseline["macro_f1"]

if mel_delta > 0.005:
    verdict = "improves"
elif mel_delta < -0.005:
    verdict = "does not improve"
else:
    verdict = "does not meaningfully change"

summary = (
    f"BCN+MNH model evaluated on the frozen BCN20000 test set ({len(test_df):,} images, "
    f"md5 {test_hash_before} — identical file as #120). Melanoma recall is {mel_recall:.1%} "
    f"versus {baseline['melanoma_recall']:.1%} for the BCN-only baseline (delta {mel_delta:+.1%}). "
    f"Melanoma false-negative rate is {mel_fnr:.1%}, meaning the model misses approximately "
    f"1 in {1/mel_fnr:.0f} melanoma cases in this test set. NMSC recall is {nmsc_recall:.1%} "
    f"versus {baseline['nmsc_recall']:.1%} baseline (delta {nmsc_recall-baseline['nmsc_recall']:+.1%}). "
    f"Macro-F1 is {macro_f1:.4f} versus {baseline['macro_f1']:.4f} baseline (delta {f1_delta:+.4f}). "
    f"Combined cancer/malignant recall is {cancer_recall:.1%} versus {baseline['cancer_recall']:.1%} "
    f"baseline (delta {cancer_recall-baseline['cancer_recall']:+.1%}). On the primary objective of "
    f"the MNH augmentation, adding histopathology-confirmed melanoma to training {verdict} melanoma "
    f"recall on this held-out test set. These results describe an educational prototype on a single-institution "
    f"dataset and do not constitute a clinical diagnostic claim or a guarantee that all cancers are detected."
)
print(summary)

OUT_SUMMARY_TXT.write_text(summary)
print(f"\nSaved: {OUT_SUMMARY_TXT}")

BCN+MNH model evaluated on the frozen BCN20000 test set (2,659 images, md5 a67861586e00812aadf46f2bdb4bc01b — identical file as #120). Melanoma recall is 61.4% versus 57.9% for the BCN-only baseline (delta +3.5%). Melanoma false-negative rate is 38.6%, meaning the model misses approximately 1 in 3 melanoma cases in this test set. NMSC recall is 75.3% versus 72.4% baseline (delta +2.9%). Macro-F1 is 0.6552 versus 0.6581 baseline (delta -0.0030). Combined cancer/malignant recall is 73.7% versus 71.9% baseline (delta +1.8%). On the primary objective of the MNH augmentation, adding histopathology-confirmed melanoma to training improves melanoma recall on this held-out test set. These results describe an educational prototype on a single-institution dataset and do not constitute a clinical diagnostic claim or a guarantee that all cancers are detected.

Saved: /Users/rehmaaziz/revela/outputs/metrics/bcn_mnh_evaluation_summary.txt
